<a href="https://colab.research.google.com/github/Rut092/rut-ai-portfolio-PHASE2-DL/blob/main/HyperParameter_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classical ML Hyper-Parameters

In [2]:
pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 6.5 MB/s eta 0:00:00


In [3]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer

In [6]:
search_space = {
    'C': Real(1e-6, 1e+6, prior='log-uniform'),   # Values are sampled uniformly from the logarithm of the range rather than the linear range.
    'gamma': Real(1e-6, 1e+1, prior='log-uniform'),
    'degree': Integer(1, 8),
    'kernel': Categorical(['linear', 'poly', 'rbf'])
}

In [8]:
bayes_search = BayesSearchCV(
    estimator=SVC(),
    search_spaces=search_space,
    n_iter=32,                # Total unique hyperparameter combinations to try
    cv=StratifiedKFold(3),    # The model is trained and validated 3 times In   each iteration, 2 folds are used for training and 1 fold for testing
    n_jobs=-1,                # Use all CPU cores
    random_state=42
)

In [9]:
bayes_search.fit(X, y)

/usr/local/lib/python3.12/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [1000000.0, np.int64(8), 1e-06, np.str_('linear')] before, using random point [283.9746656771003, np.int64(6), 4.700176809585171, 'poly']
  warnings.warn(


BayesSearchCV(cv=StratifiedKFold(n_splits=3, random_state=None, shuffle=False),
              estimator=SVC(), n_iter=32, n_jobs=-1, random_state=42,
              search_spaces={'C': Real(low=1e-06, high=1000000.0, prior='log-uniform', transform='normalize'),
                             'degree': Integer(low=1, high=8, prior='uniform', transform='normalize'),
                             'gamma': Real(low=1e-06, high=10.0, prior='log-uniform', transform='normalize'),
                             'kernel': Categorical(categories=('linear', 'poly', 'rbf'), prior=None)})

In [10]:
print(f"Best CV Score: {bayes_search.best_score_:.4f}")
print("Best Hyperparameters:")
for param, val in bayes_search.best_params_.items():
    print(f" -> {param}: {val}")

Best CV Score: 0.9667
Best Hyperparameters:
 -> C: 0.08341564384216595
 -> degree: 6
 -> gamma: 3.389034515643755
 -> kernel: linear


## Deep Learning Way

In [11]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier
from skopt import gp_minimize
from skopt.space import Real, Integer
from skopt.utils import use_named_args

In [12]:
X, y = make_classification(n_samples=1000, n_features=20, random_state=42)

In [15]:
space = [
    Integer(3, 10, name='max_depth'),
    Real(1e-3, 0.3, prior='log-uniform', name='learning_rate'),
    Integer(50, 300, name='n_estimators'),
    Real(0.5, 1.0, name='subsample')
]

In [16]:
clf = XGBClassifier(eval_metric='logloss', random_state=42)

In [17]:
# The @use_named_args decorator maps the list of values to keyword arguments
@use_named_args(space)
def objective(**params):
    # Set the current iteration's parameters into the model
    clf.set_params(**params)

    # Calculate Cross-Validation Score
    # We want to MAXIMIZE accuracy, but gp_minimize always MINIMIZES.
    # Therefore, we return the NEGATIVE accuracy.
    scores = cross_val_score(clf, X, y, cv=5, scoring='accuracy', n_jobs=-1)

    return -np.mean(scores)

In [18]:
res = gp_minimize(
    func=objective,      # The function we want to minimize
    dimensions=space,    # The parameter bounds
    n_calls=20,          # Number of sequential evaluations
    n_initial_points=5,  # Random starts to gather initial baseline data
    random_state=42
)

In [19]:
print(f"Best Optimized Accuracy: {-res.fun:.4f}")
print(f"Optimal max_depth: {res.x[0]}")
print(f"Optimal learning_rate: {res.x[1]:.4f}")
print(f"Optimal n_estimators: {res.x[2]}")
print(f"Optimal subsample: {res.x[3]:.4f}")

Best Optimized Accuracy: 0.9120
Optimal max_depth: 10
Optimal learning_rate: 0.0253
Optimal n_estimators: 194
Optimal subsample: 0.7859


## Using Optuna to do HyperParameter Tuning in Deep Learning

In [2]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.1 MB/s eta 0:00:00


In [40]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import optuna
from torch.optim import lr_scheduler
from optuna import TrialPruned

In [36]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [37]:
X,y = load_iris(return_X_y =  True)

In [53]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long) # Change to long type
X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.long)   # Change to long type


train_dataset = torch.utils.data.TensorDataset(X_train, y_train) # Corrected to include y_train
val_dataset = torch.utils.data.TensorDataset(X_val, y_val)

# Create DataLoaders
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
valloader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of validation samples: {len(val_dataset)}")

Number of training samples: 120
Number of validation samples: 30


In [54]:
print(f"Sample y_train: {y_train[:5]}")
print(f"y_train dtype: {y_train.dtype}")
print(f"y_train shape: {y_train.shape}")

print(f"\nSample y_val: {y_val[:5]}")
print(f"y_val dtype: {y_val.dtype}")
print(f"y_val shape: {y_val.shape}")

Sample y_train: tensor([0, 2, 1, 0, 1])
y_train dtype: torch.int64
y_train shape: torch.Size([120])

Sample y_val: tensor([0, 2, 1, 1, 0])
y_val dtype: torch.int64
y_val shape: torch.Size([30])


In [57]:
def objective(trial):

    input_dim = 4
    output_dim = 3

    # different trials
    hidden_dim = trial.suggest_int("hidden_dim", 32, 256, step=32)

    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5, step=0.1)

    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop", "SGD"])

    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)


    #model
    model = nn.Sequential(
        nn.Linear(input_dim, hidden_dim),
        nn.ReLU(),
        nn.Dropout(dropout_rate),
        nn.Linear(hidden_dim, output_dim)
    ).to(device)


    if optimizer_name=='Adam':
        lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name=='RMSprop': # Added RMSprop case
        lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
        optimizer = optim.RMSprop(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        lr = trial.suggest_float("lr", 1e-2, 1e-1, log=True)
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)

    scheduler_name = trial.suggest_categorical('scheduler', ['stepLR', 'ReduceLROnPlateau'])

    if scheduler_name == 'stepLR':
        # Step size means: apply gamma decay every X epochs
        step_size = trial.suggest_int('step_size', 5, 15)
        gamma = trial.suggest_float('gamma', 0.1,0.5)

        scheduler = lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)

    else:
        # Reduces LR when validation score stops improving
        patience = trial.suggest_int('patience', 5, 15)
        scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, patience=patience,mode="max", factor=0.1)


    criterion = nn.CrossEntropyLoss()

    num_epochs = 30

    for epoch in range(num_epochs):
        model.train()
        for inputs, targets in trainloader:
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()


        model.eval()
        val_correct = 0
        with torch.no_grad():

            for inputs, targets in valloader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)

                _, predicted = torch.max(outputs, 1)
                val_correct += (predicted == targets).sum().item()

        val_accuracy = val_correct / len(val_dataset)

        trial.report(val_accuracy, epoch)

        if scheduler_name == 'ReduceLROnPlateau': # Use the stored name here
            scheduler.step(val_accuracy)
        else:
            scheduler.step()

        if trial.should_prune():
            raise TrialPruned()

    return val_accuracy

In [58]:
# Create an Optuna study and optimize the objective function
study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=50) # Run 50 trials

[I 2026-06-12 13:13:29,848] A new study created in memory with name: no-name-54fdd963-1806-4379-aed9-6200895d2361
[I 2026-06-12 13:13:30,027] Trial 0 finished with value: 0.8666666666666667 and parameters: {'hidden_dim': 32, 'dropout_rate': 0.30000000000000004, 'optimizer': 'SGD', 'weight_decay': 0.0009136312866079601, 'lr': 0.016422787022658567, 'scheduler': 'ReduceLROnPlateau', 'patience': 5}. Best is trial 0 with value: 0.8666666666666667.
[I 2026-06-12 13:13:30,180] Trial 1 finished with value: 0.9333333333333333 and parameters: {'hidden_dim': 64, 'dropout_rate': 0.2, 'optimizer': 'SGD', 'weight_decay': 0.0014824583173773593, 'lr': 0.01736406666814969, 'scheduler': 'ReduceLROnPlateau', 'patience': 10}. Best is trial 1 with value: 0.9333333333333333.
[I 2026-06-12 13:13:30,373] Trial 2 finished with value: 0.9666666666666667 and parameters: {'hidden_dim': 160, 'dropout_rate': 0.0, 'optimizer': 'Adam', 'weight_decay': 0.0007818278994136994, 'lr': 0.017583062403667755, 'scheduler': 's

In [59]:
# Print the best trial's information
print("Best trial:")
trial = study.best_trial

print(f"  Value: {trial.value:.4f}")
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

Best trial:
  Value: 0.9667
  Params: 
    hidden_dim: 160
    dropout_rate: 0.0
    optimizer: Adam
    weight_decay: 0.0007818278994136994
    lr: 0.017583062403667755
    scheduler: stepLR
    step_size: 11
    gamma: 0.4513671205457106


In [60]:
print("\nIntermediate values for each trial:")
for i, trial in enumerate(study.trials):
    print(f"Trial {i}:")
    if trial.state == optuna.trial.TrialState.COMPLETE:
        print(f"  Value: {trial.value:.4f}")
        print("  Intermediate Values (epoch: accuracy):")
        for step, value in trial.intermediate_values.items():
            print(f"    Epoch {step}: {value:.4f}")
    elif trial.state == optuna.trial.TrialState.PRUNED:
        print("  This trial was pruned.")
    else:
        print(f"  Trial state: {trial.state}")


Intermediate values for each trial:
Trial 0:
  Value: 0.8667
  Intermediate Values (epoch: accuracy):
    Epoch 0: 0.7000
    Epoch 1: 0.7000
    Epoch 2: 0.7333
    Epoch 3: 0.7667
    Epoch 4: 0.7667
    Epoch 5: 0.7667
    Epoch 6: 0.7333
    Epoch 7: 0.7333
    Epoch 8: 0.8333
    Epoch 9: 0.8333
    Epoch 10: 0.8333
    Epoch 11: 0.8333
    Epoch 12: 0.8333
    Epoch 13: 0.8333
    Epoch 14: 0.8333
    Epoch 15: 0.8333
    Epoch 16: 0.8333
    Epoch 17: 0.8333
    Epoch 18: 0.8333
    Epoch 19: 0.8667
    Epoch 20: 0.8667
    Epoch 21: 0.8667
    Epoch 22: 0.8667
    Epoch 23: 0.8667
    Epoch 24: 0.8667
    Epoch 25: 0.8667
    Epoch 26: 0.8667
    Epoch 27: 0.8667
    Epoch 28: 0.8667
    Epoch 29: 0.8667
Trial 1:
  Value: 0.9333
  Intermediate Values (epoch: accuracy):
    Epoch 0: 0.8000
    Epoch 1: 0.8000
    Epoch 2: 0.8333
    Epoch 3: 0.8000
    Epoch 4: 0.8000
    Epoch 5: 0.8333
    Epoch 6: 0.8333
    Epoch 7: 0.8333
    Epoch 8: 0.8667
    Epoch 9: 0.8667
    Epoch 1